In [1]:
!pip install python-dotenv psycopg2-binary sqlalchemy pandas
!pip install psycopg2-binary


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. Cargamos el archivo de entorno desde la raíz del proyecto
load_dotenv("../.env") 

# 2. Obtenemos la URL de Render
database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise ValueError("¡CUIDADO! DATABASE_URL es None. Comprueba que tu archivo .env esté en la raíz.")

# 3. Creamos el motor de SQLAlchemy
engine = create_engine(database_url)

# 4. Definimos la función de consultas para que no dé error más adelante
def sql_query(query):
    return pd.read_sql(query, engine)

print("¡Conexión y configuración realizadas con éxito!")

¡Conexión y configuración realizadas con éxito!


In [ ]:
import pandas as pd

# 1. Cargar los archivos CSV uno a uno desde la carpeta data
# Usamos '../data/' porque el notebook está dentro de la carpeta 'src'
df_clase1 = pd.read_csv("../data/clase_1.csv", sep=";")
df_clase2 = pd.read_csv("../data/clase_2.csv", sep=";")
df_clase3 = pd.read_csv("../data/clase_3.csv", sep=";")
df_clase4 = pd.read_csv("../data/clase_4.csv", sep=";")
df_claustro = pd.read_csv("../data/claustro.csv", sep=";")

print("Archivos CSV cargados correctamente.")

# ==========================================
# 1. CREAR Y LLENAR TABLA ALUMNOS
# ==========================================
# Seleccionamos solo Nombre y Email de cada archivo
alumnos1 = df_clase1[["Nombre", "Email"]]
alumnos2 = df_clase2[["Nombre", "Email"]]
alumnos3 = df_clase3[["Nombre", "Email"]]
alumnos4 = df_clase4[["Nombre", "Email"]]

# Juntamos todas las listas de alumnos en una sola
df_alumnos = pd.concat([alumnos1, alumnos2, alumnos3, alumnos4], ignore_index=True)

# Cambiamos los nombres de las columnas para que coincidan con la base de datos
df_alumnos = df_alumnos.rename(columns={"Nombre": "nombre_alumno", "Email": "email"})

# Quitamos los alumnos duplicados basándonos en el email
df_alumnos = df_alumnos.drop_duplicates(subset=["email"])

# Guardamos en la tabla 'alumnos' de la base de datos
df_alumnos.to_sql("alumnos", engine, if_exists="replace", index=False)
print("-> Tabla 'alumnos' guardada con exito.")

# ==========================================
# 2. CREAR Y LLENAR TABLA CAMPUS
# ==========================================
# Cogemos los campus de todos los archivos
campus1 = df_clase1[["Campus"]]
campus2 = df_clase2[["Campus"]]
campus3 = df_clase3[["Campus"]]
campus4 = df_clase4[["Campus"]]
campus_claustro = df_claustro[["Campus"]]

# Juntamos y quitamos duplicados
df_campus = pd.concat([campus1, campus2, campus3, campus4, campus_claustro], ignore_index=True)
df_campus = df_campus.dropna().drop_duplicates()

# Renombramos la columna a minusculas para PostgreSQL
df_campus = df_campus.rename(columns={"Campus": "campus"})

# Guardamos en la base de datos
df_campus.to_sql("campus", engine, if_exists="append", index=False)
print("-> Tabla 'campus' guardada con exito.")

# ==========================================
# 3. CREAR Y LLENAR TABLA PROFESORES
# ==========================================
# Cogemos los profesores del archivo claustro
df_profesores = df_claustro[["Nombre", "Rol"]]
df_profesores = df_profesores.rename(columns={"Nombre": "nombre_profesor", "Rol": "rol"})

# Guardamos en la base de datos
df_profesores.to_sql("profesores", engine, if_exists="append", index=False)
print("-> Tabla 'profesores' guardada con exito.")

# ==========================================
# 4. CREAR Y LLENAR TABLA MODALIDAD
# ==========================================
# Cogemos las modalidades únicas del claustro
df_modalidad = df_claustro[["Modalidad"]].dropna().drop_duplicates()
df_modalidad = df_modalidad.rename(columns={"Modalidad": "nombre_modalidad"})

# Guardamos en la base de datos
df_modalidad.to_sql("modalidad", engine, if_exists="append", index=False)
print("-> Tabla 'modalidad' guardada con exito.")

# ==========================================
# 5. CREAR Y LLENAR TABLA VERTICALES
# ==========================================
# Cogemos las verticales únicas del claustro
df_verticales = df_claustro[["Vertical"]].dropna().drop_duplicates()
df_verticales = df_verticales.rename(columns={"Vertical": "nombre_vertical"})

# Guardamos en la base de datos
df_verticales.to_sql("verticales", engine, if_exists="append", index=False)
print("-> Tabla 'verticales' guardada con exito.")

print("\n ¡Proceso terminado! Todos los datos han sido volcados a las tablas.")

Archivos CSV cargados correctamente.
-> Tabla 'alumnos' guardada con exito.


DatabaseError: Execution failed on sql 'INSERT INTO campus (campus) VALUES (:campus)': (psycopg2.errors.UndefinedColumn) column "campus" of relation "campus" does not exist
LINE 1: INSERT INTO campus (campus) VALUES ('Madrid'), ('Valencia')
                            ^

[SQL: INSERT INTO campus (campus) VALUES (%(campus__0)s), (%(campus__1)s)]
[parameters: {'campus__0': 'Madrid', 'campus__1': 'Valencia'}]
(Background on this error at: https://sqlalche.me/e/21/f405)

In [ ]:
# Consulta : Ver alumnos
df_test_alumnos = sql_query("SELECT * FROM alumnos LIMIT 5;")
print("--- Muestra de Alumnos en PostgreSQL (Render) ---")
display(df_test_alumnos)

# Consulta : Ver tablas activas
df_test_tablas = sql_query("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public';
""")
print("--- Tablas activas en la BD de Render ---")
display(df_test_tablas)

NameError: name 'sql_query' is not defined